# 人工介入流程（Human-in-the-Loop）：前置操作門檻、風險分層與審計日誌

本課程的 README 以一段簡短的程式碼介紹人工介入流程，該程式碼讓使用者在代理人已產生回應後，選擇 `APPROVE` 或 `REJECT`。這種模式是個不錯的起點，但實際生產環境中的 HITL 實作通常需要三個額外部分：

1. 一個<strong>前置操作門檻</strong>，在代理人執行高風險步驟<strong>之前</strong>運行，以便控制成本、不可逆性和延遲。
2. <strong>風險分層</strong>，讓低風險動作能自動執行，中風險動作批次審核，只有高風險動作才需人工封鎖。
3. 一個<strong>審計日誌及修正迴圈</strong>，將每個門檻決策記錄為 JSONL，當被拒絕時，以結構化的理由重新提示代理人，而非僅印出 `Revising...`。

此筆記本在與 `06-system-message-framework.ipynb` 相同的原語基礎上構建上述功能。可在 `DEMO_MODE = True` 下全程執行（無需互動輸入），或在 `DEMO_MODE = False` 時使用真實的 `input()` 提示。注意：在 DEMO_MODE 中，第三目標的重試為腳本化，使迴圈機制可全程展示。真正的基於修正的重分類需 `DEMO_MODE = False` 且需操作者介入。

**範圍外（由其他課程處理）：** 認證與存取控制（第 06 課 README 威脅 2）、工具呼叫中介軟體（第 14 課 MAF 深度探討）、多代理辯論模式。


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## 範例 1：前置動作閘道

README 中的 HITL 範例是先呼叫代理程式，然後請使用者批准輸出。這是<strong>後置動作</strong>流程。代理程式已經執行了，所以大型語言模型（LLM）呼叫的成本已支付，而且任何副作用（發送的電子郵件、寫入的資料庫列、發表的評論）已經發生。

<strong>前置動作</strong>流程會在代理執行風險步驟前先加一道閘道。代理程式提出動作，閘道決定是否執行，只有獲得批准後副作用才會發生。

| 方面 | 後置動作批准（README 範例） | 前置動作閘道（本筆記本） |
|---|---|---|
| 何時執行批准？ | 代理程式產生輸出後 | 在任何副作用執行前 |
| 拒絕時的 LLM 成本 | 已支付 | 僅需為提案支付成本，非動作本身 |
| 不可逆的副作用 | 可能發生（動作已執行） | 被預防 |
| 審計清晰度 | 批准只是一個列印語句 | 批准會以帶時間戳、動作及原因的 JSON 記錄呈現 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 範例 3：審計日誌與修訂循環

`print("Response approved.")` 不是審計日誌。為了建立信任，每個關卡決策都應該被記錄為結構化事件，日後你可以查詢、重播，或將其附加於事件審查。

兩部分：

1. **僅追加的 JSONL。** 每個決策一行，包含時間戳、動作、層級、決策、原因。方便 grep，方便之後傳送到真正的日誌儲存。
2. **拒絕時的修訂循環。** 當關卡返回 `deny` 時，代理會帶著拒絕原因重新提示自己，讓下一個提案可以避免問題。


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 額外資源

還有幾個其他的公共專案實作了這些 HITL 模式的變體。比較這些方法以找出適合你的技術棧：

- **LangChain** 人在回路工具包裝器 ([文件](https://python.langchain.com/docs/integrations/tools/human_tools))：可直接使用的工具包裝器，會在執行中暫停以等待人工輸入。
- **AutoGen** `UserProxyAgent` ([v0.2 文件](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ 重新架構)：使用一個專門代表人在多代理對話中的代理角色。
- **Microsoft Agent Framework (MAF)** 函數調用中介軟體 ([文件](https://learn.microsoft.com/agent-framework/))：環繞每個工具/函數呼叫執行的中介軟體，適合用於閘道邏輯與批准流程。

每個專案以不同方式處理這三個子模式：LangChain 將它們包裝為工具，AutoGen 使用代理角色，而 Microsoft Agent Framework 使用函數調用中介軟體。選擇你自己的代理設計前，先完整閱讀一兩個實作。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
